In [2]:
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from collections import Counter

PROJECT_ROOT = Path.home() / "stress-detection"
save_dir=PROJECT_ROOT/"data"/"processed"
X_all=np.load(save_dir/"X_all.npy")
y_all=np.load(save_dir/"y_all.npy")


with open(save_dir/"subjects.json") as f:
    all_subjects=json.load(f)

label_names=["Baseline","Stress","Amusement","Meditation"]
SUBJECTS=sorted(set(all_subjects))

print(f"loaded X:{X_all.shape}")
print(f"loaded y:{y_all.shape}")
print(f"subjects:{SUBJECTS}")




loaded X:(1302, 42000, 4)
loaded y:(1302,)
subjects:['S11', 'S13', 'S14', 'S15', 'S16', 'S17', 'S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9']


In [3]:
import numpy as np

# Split subjects (not windows) into train/val/test
# 70% train /15% val /15% test -> 10/2/2 subjects

np.random.seed(42)
shuffled=SUBJECTS.copy()
np.random.shuffle(shuffled)

n=len(shuffled)

n_test=2
n_val=2
n_train=n-n_test-n_val

train_subjects=shuffled[:n_train]
val_subjects=shuffled[n_train:n_train+n_val]
test_subjects  = shuffled[n_train+n_val:]


print(f"Train subjects ({n_train}): {sorted(train_subjects)}")
print(f"Val   subjects ({n_val}):   {sorted(val_subjects)}")
print(f"Test  subjects ({n_test}):  {sorted(test_subjects)}")

# Build mask

Train subjects (10): ['S11', 'S13', 'S14', 'S16', 'S17', 'S4', 'S5', 'S7', 'S8', 'S9']
Val   subjects (2):   ['S3', 'S6']
Test  subjects (2):  ['S15', 'S2']


In [4]:
subjects_arr = np.array(all_subjects)

train_idx = np.where(np.isin(subjects_arr, train_subjects))[0]
val_idx   = np.where(np.isin(subjects_arr, val_subjects))[0]
test_idx  = np.where(np.isin(subjects_arr, test_subjects))[0]

# Downsample 700Hz → 70Hz (every 10th sample)
DOWNSAMPLE_FACTOR = 10
X_ds = X_all[:, ::DOWNSAMPLE_FACTOR, :]

X_train, y_train = X_ds[train_idx], y_all[train_idx]
X_val,   y_val   = X_ds[val_idx],   y_all[val_idx]
X_test,  y_test  = X_ds[test_idx],  y_all[test_idx]

N_TIMESTEPS = X_ds.shape[1]  # 4200
N_FEATURES  = X_ds.shape[2]  # 4

print(f"Downsampled shape : {X_ds.shape}  ({X_ds.nbytes/1e6:.0f} MB)")
print(f"Train : {len(X_train):>4} windows  subjects: {sorted(train_subjects)}")
print(f"Val   : {len(X_val):>4} windows  subjects: {sorted(val_subjects)}")
print(f"Test  : {len(X_test):>4} windows  subjects: {sorted(test_subjects)}")

# Free full-res data from memory
del X_all

Downsampled shape : (1302, 4200, 4)  (87 MB)
Train :  934 windows  subjects: ['S11', 'S13', 'S14', 'S16', 'S17', 'S4', 'S5', 'S7', 'S8', 'S9']
Val   :  184 windows  subjects: ['S3', 'S6']
Test  :  184 windows  subjects: ['S15', 'S2']


In [5]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

SEED = 42
torch.manual_seed(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_CLASSES = 4
CLASS_NAMES = ['Baseline', 'Stress', 'Amusement', 'Meditation']
print(f"Device: {DEVICE}")

class WESADDataset(Dataset):
    def __init__(self, X, y):
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).long()
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

def make_weighted_sampler(y):
    counts = np.bincount(y, minlength=N_CLASSES).astype(float)
    weights = 1.0 / counts
    sample_weights = weights[y]
    return WeightedRandomSampler(
        weights     = torch.from_numpy(sample_weights).float(),
        num_samples = len(y),
        replacement = True
    )

BATCH_SIZE = 16

train_ds = WESADDataset(X_train, y_train)
val_ds   = WESADDataset(X_val,   y_val)
test_ds  = WESADDataset(X_test,  y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE,
                          sampler=make_weighted_sampler(y_train), num_workers=0)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

# Sanity check
xb, yb = next(iter(train_loader))
print(f"Batch shape — X: {xb.shape}  y: {yb.shape}")
print(f"Batch label counts: {Counter(yb.numpy().tolist())}")

Device: cpu
Batch shape — X: torch.Size([16, 4200, 4])  y: torch.Size([16])
Batch label counts: Counter({1: 6, 3: 4, 2: 4, 0: 2})


In [6]:
class StressLSTM(nn.Module):
    def __init__(self, n_features, n_classes,
                 hidden_size=128, num_layers=2,
                 lstm_dropout=0.3, fc_dropout=0.4):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size    = n_features,
            hidden_size   = hidden_size,
            num_layers    = num_layers,
            batch_first   = True,
            dropout       = lstm_dropout if num_layers > 1 else 0.0,
            bidirectional = False
        )
        self.norm = nn.LayerNorm(hidden_size)
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size, 64),
            nn.ReLU(),
            nn.Dropout(fc_dropout),
            nn.Linear(64, n_classes)
        )

    def forward(self, x):
        _, (h_n, _) = self.lstm(x)
        last_hidden  = h_n[-1]           # (batch, hidden_size)
        return self.classifier(self.norm(last_hidden))

model = StressLSTM(n_features=N_FEATURES, n_classes=N_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(model)
print(f"\nTotal parameters: {total_params:,}")

# Forward pass test
with torch.no_grad():
    dummy = torch.zeros(2, N_TIMESTEPS, N_FEATURES).to(DEVICE)
    out   = model(dummy)
    print(f"Output shape: {out.shape}  ← should be (2, 4)")

StressLSTM(
  (lstm): LSTM(4, 128, num_layers=2, batch_first=True, dropout=0.3)
  (norm): LayerNorm((128,), eps=1e-05, elementwise_affine=True, bias=True)
  (classifier): Sequential(
    (0): Linear(in_features=128, out_features=64, bias=True)
    (1): ReLU()
    (2): Dropout(p=0.4, inplace=False)
    (3): Linear(in_features=64, out_features=4, bias=True)
  )
)

Total parameters: 209,476
Output shape: torch.Size([2, 4])  ← should be (2, 4)


In [7]:
from torch.optim.lr_scheduler import ReduceLROnPlateau

class_counts  = np.bincount(y_train, minlength=N_CLASSES).astype(float)
class_weights = torch.tensor(
    (1.0 / class_counts) / (1.0 / class_counts).sum() * N_CLASSES,
    dtype=torch.float32
).to(DEVICE)

print("Class weights:")
for name, w in zip(CLASS_NAMES, class_weights):
    print(f"  {name:<12}: {w:.4f}")

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer  = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler  = ReduceLROnPlateau(optimizer, mode='min', patience=5,
                                factor=0.5, min_lr=1e-5)

Class weights:
  Baseline    : 0.5101
  Stress      : 0.9270
  Amusement   : 1.7528
  Meditation  : 0.8101


In [ ]:
from sklearn.metrics import f1_score, accuracy_score

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    ctx = torch.enable_grad() if is_train else torch.no_grad()
    with ctx:
        for X_batch, y_batch in loader:
            X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
            logits = model(X_batch)
            loss   = criterion(logits, y_batch)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                optimizer.step()
            preds = logits.argmax(dim=1)
            correct    += (preds == y_batch).sum().item()
            total      += len(y_batch)
            total_loss += loss.item() * len(y_batch)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y_batch.cpu().numpy())

    return (total_loss / total,
            correct / total,
            f1_score(all_labels, all_preds, average='macro', zero_division=0))


N_EPOCHS   = 50
EARLY_STOP = 10
MODEL_DIR  = PROJECT_ROOT / 'models'
MODEL_DIR.mkdir(exist_ok=True)

history = {k: [] for k in
           ['train_loss','val_loss','train_acc','val_acc','train_f1','val_f1']}

best_val_loss, patience_counter, best_epoch = float('inf'), 0, 0

print(f"{'Ep':>4}  {'TrLoss':>8} {'TrAcc':>7} {'TrF1':>6}  "
      f"{'VlLoss':>8} {'VlAcc':>7} {'VlF1':>6}  {'LR':>8}")
print("─" * 68)

for epoch in range(1, N_EPOCHS + 1):
    tr_loss, tr_acc, tr_f1 = run_epoch(model, train_loader, criterion, optimizer)
    vl_loss, vl_acc, vl_f1 = run_epoch(model, val_loader,   criterion)
    scheduler.step(vl_loss)
    lr = optimizer.param_groups[0]['lr']

    for k, v in zip(history, [tr_loss,vl_loss,tr_acc,vl_acc,tr_f1,vl_f1]):
        history[k].append(v)

    marker = ''
    if vl_loss < best_val_loss:
        best_val_loss, best_epoch, patience_counter = vl_loss, epoch, 0
        torch.save(model.state_dict(), MODEL_DIR / 'best_model.pt')
        marker = ' ←'
    else:
        patience_counter += 1
        if patience_counter >= EARLY_STOP:
            print(f"\nEarly stop at epoch {epoch}  (best: {best_epoch})")
            break

    print(f"{epoch:>4}  {tr_loss:>8.4f} {tr_acc:>7.3f} {tr_f1:>6.3f}  "
          f"{vl_loss:>8.4f} {vl_acc:>7.3f} {vl_f1:>6.3f}  {lr:>8.2e}{marker}")

print("─" * 68)
print(f"Best epoch: {best_epoch}  val_loss: {best_val_loss:.4f}")

  Ep    TrLoss   TrAcc   TrF1    VlLoss   VlAcc   VlF1        LR
────────────────────────────────────────────────────────────────────
